In [1]:
import numpy as np
import pandas as pd
import h5py

# from Account import *
from Agent import *
from DataAsset import * 
from Exchnage import *
from Updater import *

In [2]:
import cython

In [3]:
%load_ext cython

In [19]:
%%cython
import numpy as np

def apply(account, names, current_price):
    num = current_price.shape[0]

    cdef double total_balance = 0

    cdef double price, price_yesterday, price_adjusted
    cdef double adjust_coeff
    cdef double price_diff
    cdef double shares

    cdef double returns

    cdef int i =0
    for name in names:
        if np.isnan(current_price[i, 0]):
            del account[name]
            if account._print:
                print("\x1b[31m\"%s\"\x1b[0m" % (name + '  상장폐지 !!! ###################################'))
        else:
            price = current_price[i, 0]
            price_yesterday = account[name]["현재가"]
            returns = price / price_yesterday
            if (returns > 1.35) or (returns < 0.65):
                price_diff = current_price[i, 1]
                price_adjusted = price - price_diff
                adjust_coeff = price_adjusted / price_yesterday
                if adjust_coeff == 0:
                    print(names[i])
                account[name]["평단가"] = account[name]["평단가"] * adjust_coeff
                account[name]["보유수량"] = int(account[name]["보유수량"] / adjust_coeff)

            shares = account[name]["보유수량"]
            total_balance = total_balance + price * shares
            account[name]["현재가"] = price
        i += 1
    return account, total_balance

In [20]:
import numpy as np


class AssetAccount(dict):
    """
    하나에 자산군에 대한 정보를 저장한다.(ex) 주식, 금, 외환 등)
    """

    def __init__(self, exchange, cost=(0.3, 0.03, 0), 출력=True):
        self._print = 출력
        self.type = "Account"
        self._exchange = exchange

        self._date = None
        self._asset_info = None  # (날짜, 이름, 데이터) 3개의 axis로 구성된 array
        self._total_balance = 0

        # self._balance = dict()
        self._tax, self._fee, self._slippage = cost

    def buy(self, names, 주문가격, 주문수량):
        pass

    def sell(self, names, 주문가격, 주문수량):
        pass

    def _add_assets(self, names, 주문가격, 현재가, 주문수량):
        for i in range(len(names)):
            name = names[i]
            수량 = 주문수량[i]
            가격 = 주문가격[i]

            if name not in self.keys():
                self[name] = {"현재가": 현재가[i], "평단가": 가격, "보유수량": 수량}
            else:
                self[name]["평단가"] = (수량 * 가격
                                     + self[name]["평단가"]
                                     * self[name]["보유수량"]) / (수량 + self[name]["보유수량"])
                self[name]["보유수량"] += 수량

    def _remove_assets(self, names, 주문가격, 주문수량):
        for i in range(len(names)):
            self[names[i]]["보유수량"] -= 주문수량[i]

            if self[names[i]]["보유수량"] == 0:
                del self[names[i]]

    def get_total_balance(self):
        # self._total_balance = 0
        # for name in self.keys():
        #     self._total_balance += self[name]["현재가"] * self[name]["보유수량"]

        return self._total_balance

    def _get_current_price(self):
        names = list(self.keys())
        current_price = self._exchange.get_assets_info(codes=names, fields=["현재가"])

        return names, current_price

    def _apply_current_price(self):
        names, current_price = self._get_current_price()

        self._total_balance = 0
        for i in range(len(names)):
            self[names[i]]["현재가"] = current_price[i]
            self._total_balance += self[names[i]]["현재가"] * self[names[i]]["보유수량"]

    def update_from_agent(self, date):
        self._date = date
        self._apply_current_price()

    def reset_from_agent(self, date):
        self._date = date
        for key in list(self.keys()):
            del self[key]


class StockAccount(AssetAccount):
    def __init__(self, exchange, cost=(0.3, 0.03, 0), 출력=True):
        AssetAccount.__init__(self, exchange, cost=cost, 출력=출력)

    def buy(self, names, 주문가격, 주문수량, 주문종류=None, 주문시간=None):
        체결가, 현재가 = self._exchange.buy(names, 주문가격, 주문종류=주문종류, 주문시간=주문시간)

        체결 = ~np.isnan(체결가)
        if self._print:
            for i in range(len(체결)):
                if 체결[i]:
                    print("매수 체결 : ", names[i], "체결가 : ", 체결가[i], "주문수량 : ", 주문수량[i])
                else:
                    print("매수 실패 : ", names[i], "주문가", 주문가격[i], "주문수량 : ", 주문수량[i])

        names = np.array(names)[체결]
        체결가, 현재가 = 체결가[체결].astype("float64"), 현재가[체결]
        주문수량 = np.array(주문수량)[체결]

        self._add_assets(names[주문수량 > 0], 체결가[주문수량 > 0], 현재가[주문수량 > 0], 주문수량[주문수량 > 0])

        거래대금 = np.sum(체결가 * 주문수량)
        return 거래대금

    def sell(self, names, 주문가격, 주문수량, 주문종류=None, 주문시간=None):
        체결가 = self._exchange.sell(names, 주문가격, 주문종류=주문종류, 주문시간=주문시간)

        체결 = ~np.isnan(체결가)
        if self._print == True:
            for i in range(len(체결)):
                if 체결[i]:
                    수익률 = (체결가[i] * (100 - self._tax - self._fee - self._slippage) / 100 - self[names[i]]["평단가"]) / \
                          self[names[i]]["평단가"] * 100
                    수익금 = (체결가[i] * (100 - self._tax - self._fee - self._slippage) / 100 - self[names[i]]["평단가"]) * \
                          self[names[i]]["보유수량"]

                    print("매도 체결 : ", names[i], "체결가 : ", 체결가[i], "주문수량 : ", 주문수량[i],
                          " / 수익률(%) : ", 수익률, ", 수익금(원) : ", 수익금)
                else:
                    print("매도 실패 : ", names[i], "주문가 : ", 주문가격[i], "주문수량 : ", 주문수량[i])

        names = np.array(names)[체결]
        체결가 = 체결가[체결].astype("float64")
        주문수량 = np.array(주문수량)[체결]
        self._remove_assets(names[주문수량 > 0], 체결가[주문수량 > 0], 주문수량[주문수량 > 0])

        거래대금 = np.sum((체결가 * 주문수량 * (100 - self._tax - self._fee - self._slippage) / 100).astype("int64"))
        return 거래대금

    def _get_current_price(self):
        names = list(self.keys())
        # current_price = self._exchange.get_assets_info(codes=names, fields=["현재가", "대비"])
        current_price = self._exchange._DataAsset.get_info(self._date, num=1, codes=names, fields=["현재가", "대비"]).reshape(-1, 2)

        return names, current_price

    def _apply_current_price(self):
        names, current_price = self._get_current_price()
        self._total_balance = 0
        self, self._total_balance = apply(self, names, current_price)
        
    def _apply_current_price2(self):
        names, current_price = self._get_current_price()
        self._total_balance = 0
        for i in range(len(names)):
            현재가 = current_price[i, 0]

            if np.isnan(현재가):
                del self[names[i]]
                if self._출력:
                    print("\x1b[31m\"%s\"\x1b[0m" % (names[i] + '  상장폐지 !!! ###################################'))
            else:
                전일종가 = self[names[i]]["현재가"]
                등락률 = 현재가 / 전일종가

                if (등락률 > 1.35) or ((등락률 < 0.65)):
                    대비 = current_price[i, 1]
                    수정전일종가 = 현재가 - 대비
                    수정계수 = 수정전일종가 / 전일종가
                    if 수정계수 == 0:
                        print(names[i])
                    self[names[i]]["평단가"] = self[names[i]]["평단가"] * 수정계수
                    self[names[i]]["보유수량"] = int(self[names[i]]["보유수량"] / 수정계수)

                self[names[i]]["현재가"] = 현재가
                self._total_balance += 현재가 * self[names[i]]["보유수량"]


# with pandas dataframe

In [21]:
f = h5py.File("./data/stock_info.hdf5", "r")
array_stock, axis_stock = load_data(f, "stock", chunks=5, in_memory=False)
array_vol, axis_vol = load_data(f, "변동성", chunks=5, in_memory=False)

data_stock = DataAsset(array_stock, axis_stock)
data_vol = DataAsset(array_vol, axis_vol)

updater = Updater(pd.Timestamp(2003, 1, 1), data_stock.dates)

# 거래소 생성
exchange_stock = Exchange()
exchange_stock.set_DataAsset(data_stock)

# 주식 계좌 생성
stock_account = StockAccount(exchange_stock, 출력=False)

# 거래 에이전트 생성 및 주식 계좌 등록
agent = Agent(1e10, 출력=True)
agent.set_account("stock", stock_account)

# 날짜가 변할시 업데이트 요청
updater.set_data(data_stock)
updater.set_data(data_vol)

updater.set_agent(agent)
updater.set_exchange(exchange_stock)

updater.initialization()

t = True
while updater._date != updater._list_date[-1]:
    # 가격조건
    if t:
        가격데이터 = data_stock.get_info(updater._date, num=300, fields=["현재가", "거래대금(원)"])
        t = False
    else:
        가격데이터_오늘 = data_stock.get_info(updater._date, num=1, fields=["현재가", "거래대금(원)"]).reshape(1, -1, 2)
        가격데이터 = np.concatenate((가격데이터[1:], 가격데이터_오늘), axis=0)

    거래가능 = (가격데이터[:, :, 1] > 0).all(axis=0)

    가격데이터2 = 가격데이터[:, 거래가능, :]
    종목코드 = np.array(data_stock.codes)[거래가능]

    유동성 = np.sum(가격데이터2[-20:, :, 1], axis=0)
    idx = np.argsort(유동성)[int(0.2 * len(유동성)):]  # 유동성 하위 20% 제거
    종목코드 = 종목코드[idx]

    # 변동성조건
    변동성데이터 = data_vol.get_info(updater._date, num=1, codes=종목코드)
    vol_rank = 변동성데이터.argsort(axis=0).argsort(axis=0)
    vol_rank = np.sum(vol_rank, axis=1).argsort(axis=0).argsort(axis=0)

    idx = np.argsort(vol_rank)[:int(0.2 * len(유동성))]  # 상위 20% 선정

    종목코드 = 종목코드[idx]
    배팅금액 = agent.total_balance / len(종목코드)

    리밸런스 = pd.DataFrame({"배팅금액": 배팅금액}, index=종목코드)

    # 다음날로 이동
    updater.update()

    # 종목선정
    보유주식 = agent.accounts["stock"]
    전일종가 = data_stock.get_info(updater._date, num=2, codes=보유주식.keys(), fields=["현재가"])[-2].reshape(-1)

    i = -1
    for 종목코드 in list(보유주식.keys()):
        i += 1
        주문가격 = 전일종가[i]
        보유수량 = 보유주식[종목코드]["보유수량"]

        if 종목코드 not in 리밸런스.index:
            최종수량 = 0
        else:
            최종수량 = int(리밸런스.at[종목코드, "배팅금액"] / 주문가격)

        if 보유수량 > 최종수량:
            주문량 = 보유수량 - 최종수량
            agent.sell("stock", 종목코드, 주문가격 * 1.1, max(1, min(보유수량, int(주문량 / 10))), 주문종류="조건부지정가", 주문시간="장전")
        else:
            주문량 = 최종수량 - 보유수량

            agent.sell("stock", 종목코드, 주문가격 * 1.1, int(보유수량), 주문시간="장전")
            agent.buy("stock", 종목코드, 주문가격 * 0.99, min(주문량, int(최종수량 / 10)), 주문시간="장전")

    전일종가 = data_stock.get_info(updater._date, num=2, codes=리밸런스.index, fields=["현재가"])[-2].reshape(-1)

    i = -1
    for 종목코드 in 리밸런스.index:
        i += 1
        주문가격 = 전일종가[i]

        if 종목코드 in 보유주식.keys():
            continue

        if 종목코드[1] == "9":
            continue

        주문량 = int(리밸런스.at[종목코드, "배팅금액"] / 주문가격)
        agent.buy("stock", 종목코드, 주문가격 * 0.99, max(int(주문량 / 10), 1), 주문시간="장전")

C:\Users\wkwek\AppData\Local\Programs\Python\Python36\lib\site-packages\ipykernel_launcher.py:40: RuntimeWarning: invalid value encountered in greater


장마감 :  2003-01-02 00:00:00 

 당일수익률(%) :  0.0 
 누적수익률(%) :  0.0 
 CAGR(%) 0.0 
 MDD :  0.0 
 총자산(원) :  10000000000.0 
 --------------------------------------------------

장마감 :  2003-01-03 00:00:00 

 당일수익률(%) :  0.01115646 
 누적수익률(%) :  0.011156460000005808 
 CAGR(%) 2.0568070190571275 
 MDD :  0.0 
 총자산(원) :  10001115646.0 
 --------------------------------------------------

장마감 :  2003-01-06 00:00:00 

 당일수익률(%) :  -0.04726739663180173 
 누적수익률(%) :  -0.036116210000003424 
 CAGR(%) -2.6024953654139704 
 MDD :  -0.047267396631810954 
 총자산(원) :  9996388379.0 
 --------------------------------------------------

장마감 :  2003-01-07 00:00:00 

 당일수익률(%) :  -0.06371264059107241 
 누적수익률(%) :  -0.09980584000000015 
 CAGR(%) -5.893732616390091 
 MDD :  -0.11094992191634731 
 총자산(원) :  9990019416.0 
 --------------------------------------------------

장마감 :  2003-01-08 00:00:00 

 당일수익률(%) :  -0.35795623122340486 
 누적수익률(%) :  -0.457404809999995 
 CAGR(%) -21.262532982246274 
 MDD :  -0.468509

장마감 :  2003-03-17 00:00:00 

 당일수익률(%) :  2.6687065049879237 
 누적수익률(%) :  -10.952191629999996 
 CAGR(%) -43.13656673436175 
 MDD :  -13.276520250328888 
 총자산(원) :  8904780837.0 
 --------------------------------------------------

장마감 :  2003-03-18 00:00:00 

 당일수익률(%) :  0.8583365430220976 
 누적수익률(%) :  -10.187861750000005 
 CAGR(%) -40.31222755622594 
 MDD :  -13.276520250328888 
 총자산(원) :  8981213825.0 
 --------------------------------------------------

장마감 :  2003-03-19 00:00:00 

 당일수익률(%) :  4.073692923127793 
 누적수익률(%) :  -6.529191030000003 
 CAGR(%) -27.38996766367354 
 MDD :  -13.276520250328888 
 총자산(원) :  9347080897.0 
 --------------------------------------------------

장마감 :  2003-03-20 00:00:00 

 당일수익률(%) :  1.1117131021445572 
 누적수익률(%) :  -5.4900638 
 CAGR(%) -23.220171942708777 
 MDD :  -13.276520250328888 
 총자산(원) :  9450993620.0 
 --------------------------------------------------

장마감 :  2003-03-21 00:00:00 

 당일수익률(%) :  -0.023931007584364446 
 누적수익률(%) :  -5.5

장마감 :  2003-05-27 00:00:00 

 당일수익률(%) :  1.3762307738384614 
 누적수익률(%) :  14.121714819999998 
 CAGR(%) 39.1299798127255 
 MDD :  -13.276520250328888 
 총자산(원) :  11412171482.0 
 --------------------------------------------------

장마감 :  2003-05-28 00:00:00 

 당일수익률(%) :  0.46563510795306856 
 누적수익률(%) :  14.653105589999992 
 CAGR(%) 40.428285727875824 
 MDD :  -13.276520250328888 
 총자산(원) :  11465310559.0 
 --------------------------------------------------

장마감 :  2003-05-29 00:00:00 

 당일수익률(%) :  -0.22707991960638874 
 누적수익률(%) :  14.392751410000004 
 CAGR(%) 39.323168372507844 
 MDD :  -13.276520250328888 
 총자산(원) :  11439275141.0 
 --------------------------------------------------

장마감 :  2003-05-30 00:00:00 

 당일수익률(%) :  1.0487281625915392 
 누적수익률(%) :  15.592420409999995 
 CAGR(%) 42.61191408973597 
 MDD :  -13.276520250328888 
 총자산(원) :  11559242041.0 
 --------------------------------------------------

장마감 :  2003-06-02 00:00:00 

 당일수익률(%) :  -0.8272872707467516 
 누적수익률(%)

장마감 :  2003-07-30 00:00:00 

 당일수익률(%) :  0.28638812436538774 
 누적수익률(%) :  18.835130210000006 
 CAGR(%) 34.97746909424304 
 MDD :  -13.276520250328888 
 총자산(원) :  11883513021.0 
 --------------------------------------------------

장마감 :  2003-07-31 00:00:00 

 당일수익률(%) :  0.7419482508597488 
 누적수익률(%) :  19.716825379999992 
 CAGR(%) 36.520342676764336 
 MDD :  -13.276520250328888 
 총자산(원) :  11971682538.0 
 --------------------------------------------------

장마감 :  2003-08-01 00:00:00 

 당일수익률(%) :  -0.7461163935518317 
 누적수익률(%) :  18.823598519999997 
 CAGR(%) 34.573590067767235 
 MDD :  -13.276520250328888 
 총자산(원) :  11882359852.0 
 --------------------------------------------------

장마감 :  2003-08-04 00:00:00 

 당일수익률(%) :  0.06023630060989336 
 누적수익률(%) :  18.895173460000002 
 CAGR(%) 34.15423332840648 
 MDD :  -13.276520250328888 
 총자산(원) :  11889517346.0 
 --------------------------------------------------

장마감 :  2003-08-05 00:00:00 

 당일수익률(%) :  -0.7743538389384171 
 누적수익률(%

...

장마감 :  2008-03-19 00:00:00 

 당일수익률(%) :  -0.13896417603022843 
 누적수익률(%) :  277.7481498 
 CAGR(%) 29.018105309050956 
 MDD :  -20.571910235301363 
 총자산(원) :  37774814980.0 
 --------------------------------------------------

장마감 :  2008-03-20 00:00:00 

 당일수익률(%) :  1.0383697238693927 
 누적수익률(%) :  281.67057222 
 CAGR(%) 29.256431271881468 
 MDD :  -20.571910235301363 
 총자산(원) :  38167057222.0 
 --------------------------------------------------

장마감 :  2008-03-21 00:00:00 

 당일수익률(%) :  0.27792100235293327 
 누적수익률(%) :  282.73131490000003 
 CAGR(%) 29.307735539880998 
 MDD :  -20.571910235301363 
 총자산(원) :  38273131490.0 
 --------------------------------------------------

장마감 :  2008-03-24 00:00:00 

 당일수익률(%) :  0.2950335041947204 
 누적수익률(%) :  283.86050050999995 
 CAGR(%) 29.328343244490384 
 MDD :  -20.571910235301363 
 총자산(원) :  38386050051.0 
 --------------------------------------------------

장마감 :  2008-03-25 00:00:00 

 당일수익률(%) :  -0.3299133613167914 
 누적수익률(%) :  282.59

장마감 :  2008-05-20 00:00:00 

 당일수익률(%) :  -0.9130213053636415 
 누적수익률(%) :  315.88515893000005 
 CAGR(%) 30.29154244276857 
 MDD :  -20.571910235301363 
 총자산(원) :  41588515893.0 
 --------------------------------------------------

장마감 :  2008-05-21 00:00:00 

 당일수익률(%) :  -0.4427282220738994 
 누적수익률(%) :  314.04391796000004 
 CAGR(%) 30.16679852239257 
 MDD :  -20.571910235301363 
 총자산(원) :  41404391796.0 
 --------------------------------------------------

장마감 :  2008-05-22 00:00:00 

 당일수익률(%) :  -0.10728844229585213 
 누적수익률(%) :  313.59969669 
 CAGR(%) 30.123452554953545 
 MDD :  -20.571910235301363 
 총자산(원) :  41359969669.0 
 --------------------------------------------------

장마감 :  2008-05-23 00:00:00 

 당일수익률(%) :  -0.9574178031779349 
 누적수익률(%) :  309.63981956 
 CAGR(%) 29.87423482426621 
 MDD :  -20.571910235301363 
 총자산(원) :  40963981956.0 
 --------------------------------------------------

장마감 :  2008-05-26 00:00:00 

 당일수익률(%) :  0.3191506117262386 
 누적수익률(%) :  310.947

 CAGR(%) 26.017819391506936 
 MDD :  -23.646786246591862 
 총자산(원) :  35982421661.0 
 --------------------------------------------------

장마감 :  2008-07-15 00:00:00 

 당일수익률(%) :  -0.16595005072913288 
 누적수익률(%) :  259.22708814000003 
 CAGR(%) 25.96563601316635 
 MDD :  -23.77349444354896 
 총자산(원) :  35922708814.0 
 --------------------------------------------------

장마감 :  2008-07-16 00:00:00 

 당일수익률(%) :  0.6366149701686025 
 누적수익률(%) :  261.51398156 
 CAGR(%) 26.095556575871083 
 MDD :  -23.77349444354896 
 총자산(원) :  36151398156.0 
 --------------------------------------------------

장마감 :  2008-07-17 00:00:00 

 당일수익률(%) :  -0.5074491647830032 
 누적수익률(%) :  259.67948187999997 
 CAGR(%) 25.965492669465373 
 MDD :  -23.77349444354896 
 총자산(원) :  35967948188.0 
 --------------------------------------------------

장마감 :  2008-07-18 00:00:00 

 당일수익률(%) :  1.8879076795004641 
 누적수익률(%) :  266.46989844 
 CAGR(%) 26.3764536526927 
 MDD :  -23.77349444354896 
 총자산(원) :  36646989844.0 
 ---

 총자산(원) :  33872616203.0 
 --------------------------------------------------

장마감 :  2008-09-09 00:00:00 

 당일수익률(%) :  0.553064064722016 
 누적수익률(%) :  240.59953471000003 
 CAGR(%) 24.01907509276773 
 MDD :  -30.885105075090415 
 총자산(원) :  34059953471.0 
 --------------------------------------------------

장마감 :  2008-09-10 00:00:00 

 당일수익률(%) :  -0.1821048524121162 
 누적수익률(%) :  239.97928643 
 CAGR(%) 23.966558329314182 
 MDD :  -30.885105075090415 
 총자산(원) :  33997928643.0 
 --------------------------------------------------

장마감 :  2008-09-11 00:00:00 

 당일수익률(%) :  1.9826698269713172 
 누적수익률(%) :  246.71995316 
 CAGR(%) 24.38153191340271 
 MDD :  -30.885105075090415 
 총자산(원) :  34671995316.0 
 --------------------------------------------------

장마감 :  2008-09-12 00:00:00 

 당일수익률(%) :  -5.165282547709491 
 누적수익률(%) :  228.81088793 
 CAGR(%) 23.21696957628232 
 MDD :  -30.885105075090415 
 총자산(원) :  32881088793.0 
 --------------------------------------------------

장마감 :  2008-09

KeyboardInterrupt: 

In [15]:
a = list(np.arange(1000))
b = np.arange(1000)

In [31]:
%%timeit
a.index(3)

482 ns ± 25.9 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [25]:
%%timeit
np.where(b == 3)[0][0]

2.17 µs ± 204 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [24]:
np.where(b.reshape(1, -1) == 3)[0]

array([0], dtype=int64)

In [30]:
b == 0  1

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [2]:
import cython

In [10]:
%load_ext Cython

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [17]:
def cal2():
    b = list(np.arange(3000))
    a = [b.index(x) for x in np.arange(1000)]

In [28]:
%%cython
import numpy as np


cdef cal():
    a = [b.index(x) for x in np.arange(1000)]

In [29]:
%%timeit
cal()

7 ms ± 271 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [18]:
%%timeit
cal2()

7.74 ms ± 680 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
